In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU chưa được kích hoạt trên Colab!")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
# ==================================================================================
# CELL MOUNT DRIVE DÙNG CHUNG — giữ NGUYÊN VĂN ở mọi notebook experiment
# (Baseline, Label Flipping, Backdoor, Defense). Chỉ sửa khối CẤU HÌNH bên dưới,
# và phải sửa giống hệt nhau ở mọi notebook.
# ==================================================================================
import os
import sys
import json    # dùng ở các cell sau
import csv     # dùng ở các cell sau
import shutil
import subprocess
from pathlib import Path

# ------------------------------ CẤU HÌNH (cả nhóm dùng chung) ---------------------
SHARED_FOLDER_NAME = 'data-poisoning-shared'   # tên thư mục gốc trong Drive (mọi người đặt đúng tên này)
REPO_URL = 'https://github.com/qtrong0205/data-poisoning-research'                                  # URL GitHub của project; để trống = lấy code từ <thư mục gốc>/repo
REPO_BRANCH = 'main'
LOCAL_PROJECT = Path('/content/data-poisoning-plantvillage')   # nơi chạy code (đĩa local của Colab, nhanh)
DRIVE_MOUNT = '/content/drive'
PIPELINES = ['baseline', 'label_flipping', 'backdoor', 'defenses']
# -----------------------------------------------------------------------------------

# 1. Mount Drive -----------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
elif not Path(DRIVE_MOUNT).is_dir():
    raise RuntimeError('Cell này dành cho Google Colab (hoặc đặt DRIVE_MOUNT trỏ tới thư mục Drive).')

# 2. Tìm thư mục gốc dùng chung ----------------------------------------------------------
_candidates = []
if os.environ.get('DP_DRIVE_ROOT_OVERRIDE'):          # chỉ dùng khi muốn ép đường dẫn thủ công
    _candidates.append(Path(os.environ['DP_DRIVE_ROOT_OVERRIDE']))
_candidates.append(Path(DRIVE_MOUNT) / 'MyDrive' / SHARED_FOLDER_NAME)
_candidates.append(Path(DRIVE_MOUNT) / 'Shareddrives' / SHARED_FOLDER_NAME)
_candidates.extend(sorted(Path(DRIVE_MOUNT, 'Shareddrives').glob(f'*/{SHARED_FOLDER_NAME}')))
DRIVE_ROOT = next((c for c in _candidates if c.is_dir()), None)
if DRIVE_ROOT is None:
    raise FileNotFoundError(
        f'Không tìm thấy thư mục dùng chung "{SHARED_FOLDER_NAME}". Đã tìm ở:\n  - '
        + '\n  - '.join(str(c) for c in _candidates)
        + '\nCách khắc phục: mở thư mục được chia sẻ trong Drive (Shared with me), chuột phải '
          '"Add shortcut to Drive", chọn My Drive và giữ nguyên tên, rồi chạy lại cell này.'
    )

DRIVE_DATA = DRIVE_ROOT / 'data'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'

# 3. Bảo đảm cấu trúc thư mục (chỉ tạo thư mục còn thiếu, không xóa/ghi đè gì) -----------------
for _d in [DRIVE_DATA / 'plantvillage dataset', DRIVE_DATA / 'splits', DRIVE_DATA / 'samples',
           DRIVE_DATA / 'poisoned' / 'label_flipping', DRIVE_DATA / 'poisoned' / 'backdoor',
           *(DRIVE_RESULTS / p for p in PIPELINES), *(DRIVE_CHECKPOINTS / p for p in PIPELINES)]:
    _d.mkdir(parents=True, exist_ok=True)

# Kiểm tra quyền ghi (người chỉ có quyền xem sẽ không lưu được kết quả)
_probe = DRIVE_RESULTS / f'.write_test_{os.getpid()}'
try:
    _probe.write_text('ok')
    _probe.unlink()
except OSError as exc:
    raise PermissionError(
        f'Không ghi được vào {DRIVE_RESULTS}. Hãy nhờ chủ thư mục cấp quyền Editor. ({exc})') from exc


# 4. Lấy code về đĩa local ------------------------------------------------------------------
def _sh(cmd, secret=None):
    shown = ' '.join(cmd)
    if secret:
        shown = shown.replace(secret, '***')
    print('$', shown)
    done = subprocess.run(cmd, text=True, capture_output=True)
    out = (done.stdout + done.stderr).strip()
    if secret:
        out = out.replace(secret, '***')
    if out:
        print(out)
    if done.returncode != 0:
        raise RuntimeError(f'Lệnh thất bại (exit {done.returncode}).')


if REPO_URL:
    _url, _token = REPO_URL, None
    if _url.startswith('https://') and 'github.com' in _url:
        try:                                    # repo private: lưu token trong Colab Secrets với tên GITHUB_TOKEN
            from google.colab import userdata
            _token = userdata.get('GITHUB_TOKEN')
        except Exception:
            _token = None
        if _token:
            _url = _url.replace('https://', f'https://{_token}@', 1)
    if (LOCAL_PROJECT / '.git').is_dir():
        _sh(['git', '-C', str(LOCAL_PROJECT), 'pull', '--ff-only'], secret=_token)
    else:
        _sh(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, _url, str(LOCAL_PROJECT)], secret=_token)
    CODE_SOURCE = f'GitHub {REPO_URL} ({REPO_BRANCH})'
else:
    _src = DRIVE_ROOT / 'repo'
    if not _src.is_dir():
        raise FileNotFoundError(
            f'Chưa có {_src}. Hãy upload code vào thư mục repo/ trong Drive dùng chung '
            '(chỉ code, không kèm data/results/checkpoints), hoặc điền REPO_URL.')
    _skip_anywhere = {'.git', '__pycache__', '.ipynb_checkpoints', 'results', 'checkpoints'}
    _skip_top = {'data', 'outputs', 'models'}

    def _ignore(directory, names):
        skip = {n for n in names if n in _skip_anywhere or n.endswith(('.pth', '.pt', '.onnx'))}
        if Path(directory) == _src:
            skip |= {n for n in names if n in _skip_top}
        return skip

    shutil.copytree(_src, LOCAL_PROJECT, dirs_exist_ok=True, ignore=_ignore)
    CODE_SOURCE = f'Drive: {_src}'

PROJECT_ROOT = LOCAL_PROJECT
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 5. Nối các thư mục dữ liệu/kết quả/checkpoint của project vào Drive (symlink) ---------------
#    Code không cần sửa: đường dẫn tương đối trong project tự trỏ vào Drive.
LINKS = {
    'data/plantvillage dataset': DRIVE_DATA / 'plantvillage dataset',
    'data/splits': DRIVE_DATA / 'splits',
    'data/poisoned': DRIVE_DATA / 'poisoned',
    'data/samples': DRIVE_DATA / 'samples',
    'outputs': DRIVE_RESULTS,
    'models/checkpoints': DRIVE_CHECKPOINTS,
}
for _p in PIPELINES:
    LINKS[f'experiments/{_p}/results'] = DRIVE_RESULTS / _p
    LINKS[f'experiments/{_p}/checkpoints'] = DRIVE_CHECKPOINTS / _p


def _link(rel, target):
    path = PROJECT_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_symlink():
        if Path(os.readlink(path)) == target:
            return 'OK'
        path.unlink()
    elif path.exists():
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()
        else:
            return 'BỎ QUA: đã có dữ liệu thật trong thư mục local, không ghi đè'
    os.symlink(target, path, target_is_directory=True)
    return 'OK'


_link_status = {rel: _link(rel, tgt) for rel, tgt in LINKS.items()}

# 6. Biến môi trường cho code mới (attack/defense) ---------------------------------------------
os.environ.update({
    'DP_DRIVE_ROOT': str(DRIVE_ROOT),
    'DP_DATA_ROOT': str(DRIVE_DATA),
    'DP_RESULTS_ROOT': str(DRIVE_RESULTS),
    'DP_CHECKPOINTS_ROOT': str(DRIVE_CHECKPOINTS),
})


def finish_session():
    """Gọi ở cuối phiên (hoặc trước khi bạn khác tiếp tục) để chắc chắn dữ liệu đã đồng bộ lên Drive."""
    if IN_COLAB:
        from google.colab import drive as _drive
        _drive.flush_and_unmount()
        print('Đã đồng bộ và ngắt kết nối Drive.')
    else:
        print('Không ở trong Colab: không cần đồng bộ.')


# 7. Tóm tắt -----------------------------------------------------------------------------------
print(f'Drive root : {DRIVE_ROOT}')
print(f'Code       : {PROJECT_ROOT}  <-  {CODE_SOURCE}')
print('Liên kết (project -> Drive):')
for _rel, _st in _link_status.items():
    print(f'  {_rel:<40} {_st}')
_raw = DRIVE_DATA / 'plantvillage dataset' / 'color'
_has = {
    'ảnh gốc (data/plantvillage dataset/color)': _raw.is_dir() and any(_raw.iterdir()),
    'manifest_train.csv': (DRIVE_DATA / 'splits' / 'manifest_train.csv').is_file(),
    'manifest_test.csv': (DRIVE_DATA / 'splits' / 'manifest_test.csv').is_file(),
    'class_to_idx.json': (DRIVE_DATA / 'splits' / 'class_to_idx.json').is_file(),
}
print('Dữ liệu dùng chung:')
for _name, _ok in _has.items():
    print(f'  {"CÓ   " if _ok else "THIẾU"} {_name}')
print('Quyền ghi: OK')

In [ ]:
# Kiểm tra các thư mục kết quả/checkpoint đã thật sự nối vào Drive (nếu không, kết quả sẽ MẤT khi hết phiên)
_bad = {rel: st for rel, st in _link_status.items() if st != 'OK'}
if _bad:
    raise RuntimeError(
        'Các thư mục sau chưa được nối vào Drive:\n'
        + '\n'.join(f'  {rel}: {st}' for rel, st in _bad.items())
        + '\nNguyên nhân thường gặp: thư mục đó vẫn còn file được commit trong repo (hãy git rm --cached '
          'và thêm vào .gitignore). Không chạy training khi lỗi này chưa được xử lý.')
print(f'Đã nối {len(_link_status)} thư mục của project vào Drive.')

In [ ]:
# Cài dependency, bỏ qua torch/torchvision/CUDA để không ghi đè PyTorch sẵn có của Colab
import re

SKIP_PACKAGES = {'torch', 'torchvision', 'torchaudio', 'triton'}
packages = []
for line in (PROJECT_ROOT / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    item = line.strip()
    if not item or item.startswith(('#', '-')):
        continue
    name = re.split(r'[=<>!~;\[\s@]', item, maxsplit=1)[0].lower().replace('_', '-')
    if name in SKIP_PACKAGES or name.startswith('nvidia-'):
        continue
    packages.append(item)
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print(f'Đã cài {len(packages)} package.')

In [4]:
import torch
import torchvision
import yaml
import sklearn
import matplotlib

print("Dependencies imported successfully!")

Dependencies imported successfully!


In [ ]:
# Kiểm tra dữ liệu dùng chung. Notebook KHÔNG tự tải dataset hay tạo split:
# việc đó do người quản lý làm một lần (nhiều người cùng ghi vào Drive dùng chung sẽ gây lỗi).
import pandas as pd

split_dir = PROJECT_ROOT / 'data/splits'
for _name in ('manifest_train.csv', 'manifest_test.csv', 'class_to_idx.json'):
    if not (split_dir / _name).is_file():
        raise FileNotFoundError(f'Thiếu {split_dir / _name}. Hãy báo người quản lý chuẩn bị dữ liệu.')

# So số ảnh trong manifest với số file thật của từng lớp (nhanh, chỉ liệt kê thư mục)
_problems = []
for _split in ('train', 'test'):
    _df = pd.read_csv(split_dir / f'manifest_{_split}.csv')
    for _cls, _n in _df.groupby('class_name').size().items():
        _folder = split_dir / _split / _cls
        _have = sum(1 for e in os.scandir(_folder) if e.is_file()) if _folder.is_dir() else 0
        if _have < _n:
            _problems.append({'split': _split, 'class': _cls, 'manifest': int(_n), 'on_drive': _have})
if _problems:
    display(pd.DataFrame(_problems))
    raise FileNotFoundError(f'{len(_problems)} lớp thiếu ảnh trong data/splits (bảng trên). Báo người quản lý bổ sung trước khi train.')
print('data/splits đầy đủ so với manifest.')

In [6]:
%cd /content/data-poisoning-plantvillage

def _baseline_run_complete(_mode, _seed):
    _run_root = DRIVE_RESULTS / 'baseline' / _mode / f'seed_{_seed}'
    _metrics_path = _run_root / 'metrics.json'
    _checkpoint_path = DRIVE_CHECKPOINTS / 'baseline' / _mode / f'seed_{_seed}' / 'final_model.pth'
    if not (_metrics_path.is_file() and _checkpoint_path.is_file()):
        return False
    try:
        _metrics = json.loads(_metrics_path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    return _metrics.get('mode') == _mode and _metrics.get('run_seed') == _seed

_sweep_seed = 42
if _baseline_run_complete('sweep', _sweep_seed):
    print(f'Sweep seed {_sweep_seed}: đã có kết quả hợp lệ trên Drive, bỏ qua training.')
else:
    subprocess.run(
        [sys.executable, '-m', 'experiments.baseline.experiment', '--mode', 'sweep', '--seed', str(_sweep_seed)],
        check=True,
    )

/content/data-poisoning-research
BASELINE EXPERIMENT
Device: cuda

[1/5] Splitting dataset...
Dataset successfully split into '/content/data-poisoning-research/data/splits'.

[2/5] Creating datasets and dataloaders...
Train samples: 43429
Test samples: 10876
Classes: 38

[3/5] Building model...
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100% 13.6M/13.6M [00:00<00:00, 188MB/s]

[4/5] Training baseline model...
  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 0.6079 | train_acc: 0.8361
 20% 1/5 [02:54<11:39, 174.98s/it]Epoch: 2 | train_loss: 0.5615 | train_acc: 0.8695
 40% 2/5 [05:40<08:28, 169.66s/it]Epoch: 3 | train_loss: 0.5965 | train_acc: 0.8771
 60% 3/5 [08:24<05:34, 167.01s/it]Epoch: 4 | train_loss: 0.6281 | train_acc: 0.8816
 80% 4/5 [11:18<02:49, 169.64s/it]Epoch: 5 | train_loss: 0.6232 | train_acc: 0.8856
100% 5/5 [14:08<00:00, 169.79s/it]

Checkpoint saved: /content/data-poisoning-

In [7]:
%cd /content/data-poisoning-plantvillage

_dataset_config, _ = load_config()
_confirm_seeds = _dataset_config['modes']['confirm']['seeds']
_missing_confirm_seeds = [
    _seed for _seed in _confirm_seeds
    if not _baseline_run_complete('confirm', _seed)
]
print(f'Confirm seeds đã có trên Drive: {[s for s in _confirm_seeds if s not in _missing_confirm_seeds]}')
print(f'Confirm seeds cần chạy: {_missing_confirm_seeds}')
for _seed in _missing_confirm_seeds:
    subprocess.run(
        [sys.executable, '-m', 'experiments.baseline.experiment', '--mode', 'confirm', '--seed', str(_seed)],
        check=True,
    )

print(f'Sweep: seed {_sweep_seed}; confirm: seeds {_confirm_seeds}.')
print(f'Results are stored in: {DRIVE_RESULTS / "baseline"}')
print(f'Checkpoints are stored in: {DRIVE_CHECKPOINTS / "baseline"}')


/content/data-poisoning-research
total 18M
-rw-r--r-- 1 root root 9.0M Nov  5 10:14 best_model.pth
-rw-r--r-- 1 root root 245K Nov  5 09:56 model.onnx
-rw-r--r-- 1 root root 8.7M Nov  5 09:56 model.onnx.data
9.0M	models/checkpoints/baseline/best_model.pth


In [ ]:
# Xem kết quả baseline đã có trên Drive
import json
import pandas as pd

_rows = []
for _p in sorted((DRIVE_RESULTS / 'baseline').glob('*/seed_*/metrics.json')):
    _m = json.loads(_p.read_text(encoding='utf-8'))
    _rows.append({'mode': _m['mode'], 'seed': _m['run_seed'], 'train': _m['train_samples'],
                  'epochs': _m['epochs'], **_m['evaluation']})
if _rows:
    _df = pd.DataFrame(_rows)
    display(_df)
    display(_df.drop(columns=['seed', 'train', 'epochs']).groupby('mode').agg(['mean', 'std']))
else:
    print('Chưa có kết quả nào.')

In [ ]:
# Chạy khi kết thúc phiên (sau khi đã xem xong kết quả). Sau lệnh này Drive bị ngắt:
# muốn chạy tiếp phải chạy lại cell mount (hoặc khởi động lại runtime).
finish_session()